# Generating a report with ThreeWToolkit

This direct example uses ThreeWToolkit for dataset loading, preprocessing, feature extraction, Decision Tree training, assessment, and HTML report generation.

1. Load a small 3W Dataset subset.
2. Train a Decision Tree through `SklearnTrainer`.
3. Assess test predictions.
4. Save an HTML report with metrics and a signal plot.

## 1. Load and transform the 3W Dataset

This quick workshop run uses up to 60 simulated instances from event classes 1–3. Set `MAX_INSTANCES = None` to use every matching instance.

In [14]:
from pathlib import Path

import pandas as pd
from ThreeWToolkit.assessment import ModelAssessmentConfig
from ThreeWToolkit.dataset import ParquetDatasetConfig, TransformConfig
from ThreeWToolkit.feature_extraction import (
    ConcatFeatureAdapterConfig,
    SequentialFeatureAdapterConfig,
    StatisticalConfig,
    WindowingConfig,
)
from ThreeWToolkit.models import SklearnModelsConfig
from ThreeWToolkit.preprocessing import (
    CleanSignalsConfig,
    FillLabelsConfig,
    ImputeMissingConfig,
    RemapClassConfig,
    SequentialPreprocessingAdapterConfig,
)
from ThreeWToolkit.reports import ReportGeneration
from ThreeWToolkit.trainer import SklearnTrainerConfig
from ThreeWToolkit.utils import TrainTestSplitter

RANDOM_SEED = 2026
DATASET_PATH = "../../dataset"
REPORTS_PATH = Path("output/workshop_reports")
SELECTED_CLASSES = [1] # Selecting only class 1 for faster execution

dataset = ParquetDatasetConfig(
    path=DATASET_PATH,
    split="list",
    target_class=SELECTED_CLASSES,
    event_type=["simulated"],
).build()

train_validation, test = TrainTestSplitter(
    size_training=0.8,
    size_test=0.2,
    random_state=RANDOM_SEED,
).split_data(dataset)
train, validation = TrainTestSplitter(
    size_training=0.75,
    size_test=0.25,
    random_state=RANDOM_SEED,
).split_data(train_validation)

transform = TransformConfig(
    pre_processing=SequentialPreprocessingAdapterConfig(
        steps=[
            CleanSignalsConfig(),
            ImputeMissingConfig(),
            FillLabelsConfig(),
            RemapClassConfig(),
        ]
    ),
    feature_extraction=SequentialFeatureAdapterConfig(
        steps=[
            WindowingConfig(window_size=128),
            ConcatFeatureAdapterConfig(steps=[StatisticalConfig()]),
        ]
    ),
).build()

transform.fit(train)
train_features = transform.transform(train)
validation_features = transform.transform(validation)
test_features = transform.transform(test)

print(len(train_features), len(validation_features), len(test_features))

2026-10-16 14:59:53,807 | INFO | ThreeWToolkit.dataset.parquet_dataset | Dataset found at ../../dataset
2026-10-16 14:59:53,808 | INFO | ThreeWToolkit.dataset.parquet_dataset | Validating dataset integrity...
2026-10-16 14:59:53,809 | INFO | ThreeWToolkit.dataset.parquet_dataset | Dataset integrity check passed!


68 23 23


## 2. Train and assess a Decision Tree

A shallow Decision Tree keeps this report example fast while demonstrating the complete ThreeWToolkit training and assessment workflow.

In [15]:
from sklearn.tree import DecisionTreeClassifier

trainer = SklearnTrainerConfig(
    config_model=SklearnModelsConfig(
        model_type=DecisionTreeClassifier,
        model_params={"max_depth": 8, "random_state": RANDOM_SEED},
    ),
    seed=RANDOM_SEED,
    use_class_weights=True,
    class_weight_strategy="balanced",
).build()

training_result = trainer.train(
    train_dataset=train_features,
    val_dataset=validation_features,
)
predictions = trainer.predict(test_features)

assessment = ModelAssessmentConfig(
    metrics=["accuracy", "f1"],
    export_results=False,
).build()
assessment_result = assessment.evaluate(
    training_results=training_result,
    predictions=predictions,
)

print(assessment.summary())

2026-10-16 15:00:03,083 | INFO | ThreeWToolkit.core.base_trainer | Initialized SklearnTrainer with seed=2026
2026-10-16 15:00:03,085 | INFO | ThreeWToolkit.trainer.sklearn_trainer | SklearnTrainer initialized | model=DecisionTreeClassifier | n_jobs=16
2026-10-16 15:00:03,086 | INFO | ThreeWToolkit.core.base_trainer | Starting training | train_size=68 | val_size=23
2026-10-16 15:00:03,355 | INFO | ThreeWToolkit.core.base_trainer | Dataset validation passed
2026-10-16 15:00:03,356 | INFO | ThreeWToolkit.core.base_trainer | Preparing training data...
2026-10-16 15:00:03,357 | INFO | ThreeWToolkit.trainer.sklearn_trainer | Converting dataset to arrays (size=68)
2026-10-16 15:00:12,710 | INFO | ThreeWToolkit.trainer.sklearn_trainer | Created arrays | X.shape=(39441, 150) | y.shape=(39441,)
2026-10-16 15:00:12,711 | INFO | ThreeWToolkit.core.base_trainer | Preparing validation data...
2026-10-16 15:00:12,712 | INFO | ThreeWToolkit.trainer.sklearn_trainer | Converting dataset to arrays (size=

Model Assessment Summary
Model: SklearnModels
Task Type: classification
Timestamp: 20261016_1500

Metrics:
  accuracy: 0.9021
  f1: 0.9021

Training History:
  Final train_loss: 0.9413
  Final val_loss: 0.8832


## 3. Generate the HTML report

The report receives the trained toolkit model, test predictions, assessment metrics, dataset sizes, and one original sensor signal.

In [16]:
report = ReportGeneration(
    model=assessment_result.model,
    train_len=len(train_features),
    test_len=len(test_features),
    predictions=pd.Series(assessment_result.predictions),
    calculated_metrics=assessment_result.metrics or {},
    plot_config=None, 
    title="3W Model Assessment Report",
    author="3W Toolkit Workshop 2026",
    reports_dir=REPORTS_PATH,
    export_report_after_generate=True,
)
html = report.generate_summary_report(format="html")

print(f"Generated HTML: {len(html):,} characters")

2026-10-16 15:00:20,175 | INFO | ThreeWToolkit.reports.report_generation | Reports will be saved to directory: 'output/workshop_reports/report-3W Model Assessment Report'
2026-10-16 15:00:20,176 | INFO | ThreeWToolkit.reports.report_generation | Generating HTML report from template: 'report_template.html'...
2026-10-16 15:00:20,186 | INFO | ThreeWToolkit.reports.report_generation | Markdown report generated successfully.
2026-10-16 15:00:20,186 | INFO | ThreeWToolkit.reports.report_generation | Saving HTML report to 'output/workshop_reports/report-3W Model Assessment Report/html/report-3W Model Assessment Report.html'...
2026-10-16 15:00:20,188 | INFO | ThreeWToolkit.reports.report_generation | HTML report saved successfully to 'output/workshop_reports/report-3W Model Assessment Report/html/report-3W Model Assessment Report.html'


Generated HTML: 2,488 characters
